# Incident Report Structuring -- LoRA Fine-Tuning

Fine-tunes a small open-weight model (Qwen3-4B) via LoRA to extract
structured JSON from freeform incident reports, matching the schema in
`schema.py` at the repo root.

**Runs on Google Colab's free T4 GPU tier.** No payment required.

Before running: upload `data/train.jsonl`, `data/val.jsonl`, and `schema.py` (from the repo root) (generated
by `data/generate_dataset.py`) to this Colab session, or mount Google
Drive and point the paths below at them.


## 1. Install dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q pydantic


## 2. Load the base model

`unsloth/Qwen3-4B-bnb-4bit` is Unsloth's pre-quantized 4-bit version --
fits comfortably on a free T4 with room to spare for LoRA training.

Swap to `unsloth/Qwen3-1.7B-bnb-4bit` instead if you want faster
iteration at the cost of some capability, or
`unsloth/Qwen2.5-1.5B-bnb-4bit` if you specifically want the earlier
model generation for comparison purposes.


In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 512  # reports are short (60-120 words) + JSON output is compact
MODEL_NAME = "unsloth/Qwen3-4B-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,       # auto-detect (bfloat16 on T4)
    load_in_4bit=True,
)


## 3. Add LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,         # 0 is optimized in Unsloth
    bias="none",            # "none" is optimized in Unsloth
    use_gradient_checkpointing="unsloth",  # for longer-context / lower VRAM
    random_state=42,
    use_rslora=False,
)


## 4. Prepare the dataset

Each training example becomes a 3-turn chat: a system message describing
the schema and instructing JSON-only output, the freeform report as the
user turn, and the gold-label JSON as the assistant turn. The model
learns to reproduce the assistant turn.


In [ ]:
import json

SYSTEM_PROMPT = """You are an incident-report structuring assistant. Given a freeform incident report, extract the following fields as a single JSON object and output ONLY that JSON object, nothing else:

- severity: one of "critical", "high", "medium", "low"
- category: one of "security", "infrastructure", "application", "data", "network"
- affected_system: the name of the affected system as mentioned in the report
- reported_by: one of "on-call engineer", "SRE", "customer support agent", "security analyst", "devops engineer", "qa tester", "unspecified"
- time_detected: one of "just now", "within the last hour", "this morning", "this afternoon", "overnight", "yesterday", "unspecified"
- action_taken: one of "none", "restarted the service", "rolled back the deployment", "escalated to on-call", "applied a temporary workaround", "disabled the affected feature", "notified the vendor"
- requires_escalation: true or false
- customer_impact: true or false
"""

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def to_chat_example(row):
    return {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["report_text"]},
            {"role": "assistant", "content": json.dumps(row["label"])},
        ]
    }

train_rows = [to_chat_example(r) for r in load_jsonl("train.jsonl")]
val_rows = [to_chat_example(r) for r in load_jsonl("val.jsonl")]
print(f"{len(train_rows)} train examples, {len(val_rows)} val examples")


In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

def formatting_func(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False, enable_thinking=False)
        for convo in examples["conversations"]
    ]
    return {"text": texts}

train_dataset = Dataset.from_list(train_rows).map(formatting_func, batched=True)
val_dataset = Dataset.from_list(val_rows).map(formatting_func, batched=True)

print(train_dataset[0]["text"][:500])


## 5. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()


## 6. Save the adapter

Saves locally in this Colab session. Download the `lora_adapter` folder
(or push to the Hugging Face Hub, recommended -- makes it loadable
directly from `inference/predictor.py` without re-uploading files by
hand) before your Colab session ends, since Colab storage is ephemeral.


In [ ]:
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

# Optional: push to the Hugging Face Hub (free, public model hosting) so
# inference/predictor.py and app/gradio_app.py can load it by repo id
# instead of needing the files copied locally.
#
# from huggingface_hub import login
# login()  # paste a free HF access token when prompted
# model.push_to_hub("your-username/incident-report-qwen3-lora")
# tokenizer.push_to_hub("your-username/incident-report-qwen3-lora")


## 7. Quick smoke test

In [ ]:
FastLanguageModel.for_inference(model)  # 2x faster inference mode

test_report = (
    "Customers are reporting they can't log in since about 30 minutes ago. "
    "Looks like the auth-service is returning timeouts. Security team is not "
    "involved yet, this looks like a straightforward infra issue. I've already "
    "restarted the service once, didn't help. Escalating now."
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_report},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    enable_thinking=False,  # Qwen3 defaults to a <think>...</think> reasoning
                            # block otherwise -- not needed for a simple
                            # extraction task, and breaks naive JSON parsing
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.1, do_sample=True)
decoded = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(decoded)

# Uses the same robust parser the eval harness and inference wrapper use
# (strips markdown fences, extracts the first {...} block even with stray
# text around it) rather than a naive json.loads. Requires schema.py to
# have been uploaded to this Colab session alongside train.jsonl/val.jsonl.
from schema import parse_model_output

parsed = parse_model_output(decoded)
if parsed is not None:
    print("\nValid, schema-conforming JSON:")
    print(parsed.model_dump_json(indent=2))
else:
    print("\n[!] Did not parse as a valid IncidentReport.")
